In [1]:
# Auto-generated by cram CLI — do not hand-edit.
# Rebuild with: python tools/cram.py build

import sys
from pathlib import Path

# Portably discover repo root containing 'engine' and add to sys.path
_cur = Path.cwd().resolve()
while _cur != _cur.parent:
    if (_cur / 'engine' / '__init__.py').is_file():
        if str(_cur) not in sys.path:
            sys.path.insert(0, str(_cur))
        break
    _cur = _cur.parent

# In WebAssembly / Pyodide / JupyterLite environments, pre-load dependencies
if 'pyodide' in sys.modules or hasattr(sys, '_emscripten_info') or sys.platform == 'emscripten':
    try:
        import piplite
        await piplite.install(['pyyaml', 'matplotlib', 'ipywidgets', 'numpy'])
    except Exception:
        pass

# Explicit imports for Pyodide AST scanner and interactive widget ecosystem
try:
    import yaml
except ImportError:
    pass
import matplotlib.pyplot as plt
import ipywidgets as widgets

from engine import Worksheet, apply_default_style

apply_default_style()

ws = Worksheet.load('lesson_22_ising_model_and_spin_kinetics.yaml')
ws.start()


$\def\AA{\text{Å}}$

# MSE 485 · Lesson 22: The Ising Model — Exact Solutions, Phase Transitions, Spin Kinetics & Cluster Monte Carlo

_Session 9c44d588 · Estimated time: ~120 minutes · 42 problems_

**Recording your answers.** Submit each problem as you go. Your answers are saved to the workspace as you submit, *and* the running record at the bottom of the notebook keeps your progress. When you finish, run the last two cells — **`show_record()`** and **`finalize()`** — to record your participation. **If you reopened this workspace** (or the kernel restarted), use **Kernel → Restart Kernel and Run All Cells** before finalizing, so every answer is reloaded.

### Learning Objectives
- Formulate the Ising Hamiltonian, prove discrete Z_2 spin-inversion symmetry, and derive exact isomorphisms to lattice gases (liquid-gas coexistence) and binary alloys (order-disorder transitions).
- Solve the 1D Ising chain using the transfer matrix method, calculate the spin-spin correlation function, and apply the Peierls kink argument to prove the absence of finite-temperature phase transitions in 1D.
- Analyze Onsager's exact 2D square-lattice solution, Kramers-Wannier duality, the critical temperature k_B T_c / J, and the 2D critical exponents governing spontaneous magnetization, susceptibility, and specific heat.
- Investigate the finite-size order parameter distribution P_L(M), quantify the interfacial free energy barrier Delta F = 2 sigma L^(d-1), and apply Binder cumulant crossings to locate critical points.
- Differentiate non-conserved Glauber dynamics (Model A, curvature-driven domain growth) from conserved Kawasaki dynamics (Model B, diffusion-limited spinodal decomposition and Ostwald ripening).
- Quantify critical slowing down tau ~ xi^z, derive the Fortuin-Kasteleyn random cluster representation, and implement non-local Swendsen-Wang and Wolff cluster Monte Carlo algorithms.


In [2]:
ws.render_section('sec_ising_hamiltonian_and_mappings')

## 1. The Ising Hamiltonian, Global Z_2 Symmetry, Lattice Gases, and Binary Alloys

The Ising model represents discrete scalar degrees of freedom (magnetic moments or atomic species) residing on the sites of a $d$-dimensional crystal lattice. Each site $i \in \{1, 2, \dots, N\}$ carries a spin variable $s_i \in \{-1, +1\}$. In the presence of a uniform external magnetic field $h$, the total energy of a configuration $\{s_i\}$ is governed by the Hamiltonian:
$$\mathcal{H}(\{s_i\}) = -J \sum_{\langle i, j \rangle} s_i s_j - h \sum_{i=1}^N s_i$$
where $\langle i, j \rangle$ denotes a sum over all distinct nearest-neighbor pairs, and $J$ is the exchange coupling constant. When $J > 0$, neighboring spins lower their energy by aligning parallel (ferromagnetism); when $J < 0$, antiparallel alignment is favored (antiferromagnetism).

### Global Discrete Symmetry and Spontaneous Symmetry Breaking
In the absence of an external field ($h = 0$), the Hamiltonian is invariant under the global spin inversion operation:
$$s_i \to -s_i \quad \forall i \in \{1, \dots, N\} \implies \mathcal{H}(\{-s_i\}) = \mathcal{H}(\{s_i\})$$
This invariance constitutes a discrete $\mathbb{Z}_2$ symmetry. At high temperatures ($T > T_c$), thermal fluctuations preserve this symmetry in the thermodynamic limit, producing zero net magnetization:
$$\langle M \rangle = \frac{1}{N}\sum_{i=1}^N \langle s_i \rangle = 0$$
Below a critical temperature $T_c$ (for dimensions $d \ge 2$), the system undergoes **spontaneous symmetry breaking**, collapsing into one of two degenerate macroscopic phases characterized by non-zero spontaneous magnetization $\pm M_0(T)$.

### Isomorphism 1: The Lattice Gas (Liquid-Gas Coexistence)
The Ising model maps rigorously onto a classical lattice model of a simple fluid. Let site $i$ have occupation number $n_i \in \{0, 1\}$, where $n_i = 1$ denotes the presence of a fluid molecule and $n_i = 0$ denotes an empty site (vacancy). Multiple occupancy is prohibited by hard-core exclusion. If neighboring occupied sites experience an attractive dispersion well $-\epsilon$ ($\epsilon > 0$) and the system is held at chemical potential $\mu$, the grand canonical Hamiltonian is:
$$\mathcal{H}_{\text{LG}} = -\epsilon \sum_{\langle i, j \rangle} n_i n_j - \mu \sum_{i=1}^N n_i$$
Applying the linear coordinate transformation $s_i = 2n_i - 1 \iff n_i = \frac{1 + s_i}{2}$ maps $\mathcal{H}_{\text{LG}}$ directly onto the Ising Hamiltonian with parameters:
$$J = \frac{\epsilon}{4}, \qquad h = \frac{\mu - \mu_0}{2}, \qquad \mu_0 = -\frac{z\epsilon}{2}$$
where $z$ is the lattice coordination number. The magnetic transition at $h = 0$ corresponds to liquid-gas phase coexistence along the saturation line $\mu = \mu_0$. The spontaneous magnetization $M$ directly represents the density discontinuity across the first-order liquid-gas coexistence curve:
$$\rho - \rho_c = \frac{1}{2} M, \qquad \rho_c = \frac{1}{2}$$

### Isomorphism 2: Binary Alloys (Order-Disorder Transitions)
In a crystalline binary alloy comprising atomic species $A$ and $B$ (such as Cu-Au or Fe-Al), let $s_i = +1$ represent an $A$ atom and $s_i = -1$ represent a $B$ atom on site $i$. The nearest-neighbor interaction energies are $V_{AA}$, $V_{BB}$, and $V_{AB}$. The configurational energy is isomorphic to the Ising model with effective exchange parameter:
$$J = \frac{1}{4}(2 V_{AB} - V_{AA} - V_{BB})$$
- If $J < 0$ (i.e., $2 V_{AB} > V_{AA} + V_{BB}$), like bonds ($A-A$ and $B-B$) are energetically favored, driving low-temperature **phase separation** (clustering/precipitation) into $A$-rich and $B$-rich domains.
- If $J > 0$ (i.e., $2 V_{AB} < V_{AA} + V_{BB}$), unlike bonds ($A-B$) are energetically favored, driving low-temperature **chemical ordering** into an interpenetrating superlattice (antiferromagnetic order).


In [3]:
ws.render_section('q22_01_sublattice_gauge_transformation')

**Q1 · medium** · 💡

Consider an antiferromagnetic Ising model with coupling $J_{\text{AFM}} = -J < 0$ on a bipartite lattice (e.g., a simple cubic or square lattice) partitioned into interpenetrating sublattices $\mathcal{A}$ and $\mathcal{B}$, in zero external magnetic field. Which exact local transformation maps this antiferromagnet onto an equivalent ferromagnetic Ising Hamiltonian with coupling $+J$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Examine the topology of a bipartite graph: every nearest-neighbor bond connects a site in one sublattice to a site in the other sublattice. Consider how an operation acting exclusively on one sublattice alters the sign of each nearest-neighbor product.

</div>
</details>

- [ ] **(A)** A sublattice spin-inversion gauge transformation: invert all spins on sublattice $\mathcal{B}$ ($s_j \to -s_j$ for $j \in \mathcal{B}$) while keeping spins on sublattice $\mathcal{A}$ unchanged.
- [ ] **(B)** A global uniform spin inversion: invert every spin on the entire lattice simultaneously ($s_i \to -s_i$ for all $i$).
- [ ] **(C)** A $90^\circ$ spatial rotation of the lattice coordinate axes that transposes nearest neighbors into next-nearest neighbors.
- [ ] **(D)** An imaginary Wick rotation of the inverse temperature variable: $\beta \to i\beta$.

In [4]:
ws.render_section('q22_02_geometric_frustration')

**Q2 · medium** · 💡

If the same nearest-neighbor antiferromagnetic Ising interaction ($J < 0$) is placed on a triangular lattice rather than a square lattice, the bipartite sublattice mapping fails completely, leading to geometric frustration. What is the fundamental physical consequence of this frustration on the $T = 0$ ground state?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Attempt to assign discrete binary spin values to the three vertices of a single elementary triangle such that every edge simultaneously minimizes the antiferromagnetic exchange energy, and consider the multiplicity of configurations.

</div>
</details>

- [ ] **(A)** The ground state possesses extensive macroscopic degeneracy, resulting in non-zero residual entropy $S_0 > 0$ at absolute zero and suppressing conventional long-range Néel order.
- [ ] **(B)** The system collapses into a unique collinear ferromagnetic state to minimize total bond strain.
- [ ] **(C)** The critical temperature $T_c$ shifts to infinity because triangular loops amplify thermal correlations.
- [ ] **(D)** The discrete $\mathbb{Z}_2$ symmetry is dynamically broken into continuous $\mathrm{SU}(2)$ gauge symmetry.

In [5]:
ws.render_section('q22_03_lattice_gas_coexistence_chemical_potential')

**Q3 · hard** · 💡

In the lattice gas mapping, fluid molecules occupy lattice sites with attractive nearest-neighbor interaction $-\epsilon$ and chemical potential $\mu$. Liquid-gas phase coexistence at temperatures below the critical point corresponds to the zero-field line ($h = 0$) in the Ising ferromagnet. What is the analytical value of the coexistence chemical potential $\mu_0$ on a lattice with coordination number $z$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Substitute the spin transformation into the lattice gas grand canonical energy expression and identify the linear single-site term that corresponds to the external Zeeman field h in the Ising Hamiltonian.

</div>
</details>

- [ ] **(A)** $\mu_0 = -\frac{z\epsilon}{2}$
- [ ] **(B)** $\mu_0 = -z\epsilon$
- [ ] **(C)** $\mu_0 = +\frac{z\epsilon}{4}$
- [ ] **(D)** $\mu_0 = 0$

In [6]:
ws.render_section('q22_04_binary_alloy_ordering_condition')

**Q4 · medium** · 💡

In a crystalline binary alloy with nearest-neighbor bond energies $V_{AA}$, $V_{BB}$, and $V_{AB}$, the configurational energy maps to an Ising model with effective exchange parameter $J = \frac{1}{4}(2 V_{AB} - V_{AA} - V_{BB})$. Which thermodynamic behavior occurs when $2 V_{AB} < V_{AA} + V_{BB}$ at low temperatures?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Evaluate the sign of the effective coupling parameter J under this inequality, and consider whether bonds between like atoms or bonds between unlike atoms are energetically preferred.

</div>
</details>

- [ ] **(A)** The alloy forms an ordered interpenetrating superlattice (chemical ordering) because unlike A–B pairs have lower energy than like A–A and B–B pairs.
- [ ] **(B)** The alloy phase-separates into macroscopic A-rich and B-rich domains (miscibility gap) because like pairs are energetically favored.
- [ ] **(C)** The alloy remains an ideal random solid solution down to absolute zero with zero heat of mixing.
- [ ] **(D)** The lattice becomes mechanically unstable and undergoes a martensitic shear collapse.

In [7]:
ws.render_section('q22_05_ground_state_energy_numeric')

**Q5 · easy** · 💡

A square-lattice Ising ferromagnet containing $N = 250$ sites with periodic boundary conditions has nearest-neighbor exchange coupling $J = 2.4\text{ meV}$ and zero external magnetic field. Determine the total ground state energy $E_{\text{ground}}$ in meV when all spins are fully aligned.

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Consider how many total nearest-neighbor bonds exist on a square lattice with periodic boundaries where each site has four nearest neighbors, and account for the energy contributed by each parallel bond.

</div>
</details>

_Numeric answer required (in meV)._

In [8]:
ws.render_section('q22_06_single_spin_flip_energy_numeric')

**Q6 · medium** · 💡

For the ferromagnet in the preceding problem ($N = 250$, $J = 2.4\text{ meV}$, square lattice), starting from the fully aligned ground state, a single spin in the interior of the crystal is flipped antiparallel to its neighbors. Calculate the resulting change in system energy $\Delta E$ in meV.

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Identify how many nearest-neighbor bonds have their relative spin alignment altered by flipping one interior spin, and evaluate the difference in bond energy between an antiparallel pair and a parallel pair.

</div>
</details>

_Numeric answer required (in meV)._

In [9]:
ws.render_section('q22_07_lattice_gas_compressibility_relation')

**Q7 · hard** · 💡

Under the lattice gas isomorphism, how does the isothermal compressibility $\kappa_T = -\frac{1}{V}\left(\frac{\partial V}{\partial P}\right)_T = \frac{1}{\rho^2}\left(\frac{\partial \rho}{\partial \mu}\right)_T$ of the fluid behave as the liquid-gas critical point is approached along the critical isochore?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Relate the density derivative with respect to chemical potential to the magnetic susceptibility chi_T = (partial M / partial h)_T in the Ising model, and recall the behavior of magnetic susceptibility at second-order phase transitions.

</div>
</details>

- [ ] **(A)** It diverges to infinity with critical exponent $\gamma$, mirroring the divergence of the magnetic susceptibility $\chi_T$ in the Ising ferromagnet.
- [ ] **(B)** It drops discontinuously to zero because liquid and vapor densities equalize.
- [ ] **(C)** It remains strictly constant and equal to $\frac{1}{\rho_c k_B T}$.
- [ ] **(D)** It develops an imaginary component associated with mechanical cavitation.

In [10]:
ws.render_section('sec_exact_solutions_and_transfer_matrix')

### The 1D Ising Chain and the Transfer Matrix Method
Consider a linear chain of $N$ spins with periodic boundary conditions ($s_{N+1} = s_1$) in a uniform field $h$. The Hamiltonian is:
$$\mathcal{H} = -J \sum_{i=1}^N s_i s_{i+1} - \frac{h}{2} \sum_{i=1}^N (s_i + s_{i+1})$$
The canonical partition function factorizes over the bonds:
$$Z_N = \sum_{s_1=\pm 1} \dots \sum_{s_N=\pm 1} \prod_{i=1}^N T(s_i, s_{i+1}) = \text{Tr}\left(\mathbf{T}^N\right)$$
where $\mathbf{T}$ is the $2 \times 2$ **transfer matrix** with elements $T(s, s') = \exp\left[\beta J s s' + \frac{1}{2}\beta h (s + s')\right]$:
$$\mathbf{T} = \begin{pmatrix} e^{\beta(J + h)} & e^{-\beta J} \\ e^{-\beta J} & e^{\beta(J - h)} \end{pmatrix}$$
In zero external field ($h = 0$), the eigenvalues are found from $\det(\mathbf{T} - \lambda \mathbf{I}) = 0$:
$$\lambda_1 = 2\cosh(\beta J), \qquad \lambda_2 = 2\sinh(\beta J)$$
Since $\beta J > 0$, $\lambda_1 > \lambda_2 > 0$ strictly for all finite temperatures ($T > 0$). In the thermodynamic limit ($N \to \infty$), the partition function and free energy per spin are determined entirely by the largest eigenvalue:
$$f(T, 0) = -\lim_{N \to \infty} \frac{k_B T}{N} \ln Z_N = -k_B T \ln \lambda_1 = -k_B T \ln(2\cosh\beta J)$$

### Spin-Spin Correlation Function and Correlation Length
The zero-field spin-spin correlation function between two sites separated by distance $r$ lattice units is computed by inserting Pauli $\sigma_z$ operators into the trace:
$$\langle s_i s_{i+r} \rangle = \frac{\lambda_1^{N-r} \lambda_2^r + \lambda_2^{N-r} \lambda_1^r}{\lambda_1^N + \lambda_2^N} \xrightarrow{N \to \infty} \left(\frac{\lambda_2}{\lambda_1}\right)^r = (\tanh\beta J)^r$$
Writing this in the standard exponential decay form $\langle s_i s_{i+r} \rangle = \exp(-r / \xi)$ yields the exact correlation length:
$$\xi(T) = -\frac{1}{\ln(\tanh\beta J)}$$
At low temperatures ($T \to 0$, $\beta J \to \infty$), $\tanh(\beta J) \approx 1 - 2e^{-2\beta J}$, giving $\xi(T) \approx \frac{1}{2} e^{2\beta J}$. The correlation length diverges only at $T = 0$, confirming that long-range order cannot persist at any non-zero temperature in one dimension.

### The Peierls Kink (Domain Wall) Argument in 1D
Why can no 1D system with short-range interactions order at $T > 0$? Consider a fully aligned ferromagnetic ground state ($s_i = +1$ for all $i$). Inserting a single domain wall (kink) that flips all spins to the right of position $k$ into $s_i = -1$ disrupts exactly one nearest-neighbor bond:
- **Energetic Cost:** $\Delta E = 2J$ (a finite, constant energy independent of system size $N$).
- **Entropic Gain:** The domain wall can be placed between any of the $N$ pairs of adjacent sites. The configurational entropy is $\Delta S = k_B \ln N$.
- **Free Energy Change:**
$$\Delta F = \Delta E - T \Delta S = 2J - k_B T \ln N$$
For any temperature $T > 0$, as $N \to \infty$, the entropic term $-k_B T \ln N \to -\infty$ completely overwhelms the finite energetic barrier $\Delta E = 2J$. Therefore, domain walls spontaneously proliferate at any non-zero temperature, destroying long-range order.


In [11]:
ws.render_section('q22_08_transfer_matrix_eigenvalues')

**Q8 · medium** · 💡

In the transfer matrix solution of the 1D Ising model with periodic boundary conditions, why does the free energy per spin $f(T) = -k_B T \lim_{N \to \infty} \frac{1}{N}\ln Z_N$ depend exclusively on the largest eigenvalue $\lambda_1$ in the thermodynamic limit?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Factor lambda_1^N out of the partition function expression Z_N = lambda_1^N + lambda_2^N and examine the behavior of the remaining term as N approaches infinity.

</div>
</details>

- [ ] **(A)** Because factoring $Z_N = \lambda_1^N \left[1 + \left(\frac{\lambda_2}{\lambda_1}\right)^N\right]$ reveals that $(\lambda_2 / \lambda_1)^N$ vanishes exponentially as $N \to \infty$ when $\lambda_1 > \lambda_2$.
- [ ] **(B)** Because the trace of a matrix is mathematically identical to its largest eigenvalue.
- [ ] **(C)** Because the second eigenvalue $\lambda_2$ is strictly zero at all temperatures.
- [ ] **(D)** Because quantum tunneling forces all lower eigenstates to collapse.

In [12]:
ws.render_section('q22_09_correlation_function_eigenvalue_ratio')

**Q9 · medium** · 💡

For the zero-field 1D Ising chain, the spin-spin correlation function between two spins separated by $r$ lattice sites is $\langle s_i s_{i+r} \rangle = (\lambda_2 / \lambda_1)^r$. In terms of the exchange coupling $J$ and temperature $T$, what is the exact algebraic ratio $\lambda_2 / \lambda_1$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Evaluate the ratio of the two zero-field eigenvalues lambda_2 = 2 sinh(beta J) and lambda_1 = 2 cosh(beta J).

</div>
</details>

- [ ] **(A)** $\tanh(\beta J)$
- [ ] **(B)** $\coth(\beta J)$
- [ ] **(C)** $\exp(-2\beta J)$
- [ ] **(D)** $\operatorname{sech}(\beta J)$

In [13]:
ws.render_section('q22_10_correlation_length_numeric')

**Q10 · medium** · 💡

For a 1D Ising chain in zero magnetic field governed by nearest-neighbor coupling $J$, transfer matrix analysis establishes that the spin-spin correlation function decays exponentially as $\langle s_i s_{i+r} \rangle = e^{-r / \xi}$. At an elevated temperature where the nearest-neighbor spin correlation parameter is measured to be $\tanh(\beta J) = 0.82$, calculate the spatial correlation length $\xi$ in units of the lattice spacing.

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Relate the spatial decay rate of the spin correlation function across r lattice sites to the natural logarithm of the transfer matrix eigenvalue ratio lambda_2 / lambda_1 = tanh(beta J).

</div>
</details>

_Numeric answer required._

In [14]:
ws.render_section('q22_11_peierls_kink_argument')

**Q11 · hard** · 💡

In the Peierls argument for the 1D Ising chain, why does the free energy change to create a single domain boundary, $\Delta F = 2J - k_B T \ln N$, guarantee the destruction of spontaneous magnetization at any finite temperature $T > 0$ as $N \to \infty$, whereas the corresponding argument in $d = 2$ allows stable ferromagnetic order?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compare how the energetic cost of an enclosing domain boundary scales with its geometric perimeter/area versus how the number of possible boundary configurations (entropy) scales in one dimension versus two dimensions.

</div>
</details>

- [ ] **(A)** In 1D, the energetic barrier $\Delta E = 2J$ is independent of system size while entropy grows as $\ln N$, driving $\Delta F \to -\infty$. In 2D, enclosing a domain of linear size $L$ costs energy scaling with perimeter ($\Delta E \sim 2\sigma L$), which overwhelms boundary configurational entropy at sufficiently low temperatures.
- [ ] **(B)** In 1D, spins are prohibited by quantum uncertainty from having discrete values, whereas 2D spins are classical vectors.
- [ ] **(C)** In 2D, domain walls cannot form closed loops due to topological conservation laws.
- [ ] **(D)** The 1D transfer matrix has imaginary eigenvalues, which destabilizes boundary walls.

In [15]:
ws.render_section('q22_12_specific_heat_schottky_anomaly')

**Q12 · medium** · 💡

How does the zero-field specific heat capacity $C_V(T) = -T \frac{\partial^2 f}{\partial T^2}$ of the 1D Ising chain behave as a function of temperature?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Differentiate the 1D free energy per spin f = -k_B T ln(2 cosh(beta J)) twice with respect to temperature and observe whether any mathematical singularity arises at non-zero T.

</div>
</details>

- [ ] **(A)** It exhibits a broad, perfectly analytic Schottky-like peak that drops smoothly to zero both as $T \to 0$ and as $T \to \infty$, with no mathematical singularity at any non-zero temperature.
- [ ] **(B)** It diverges logarithmically $C_V \sim \ln|T - T_c|$ at a finite critical temperature.
- [ ] **(C)** It undergoes a discontinuous jump of $\Delta C_V = \frac{3}{2} k_B$ as predicted by mean-field theory.
- [ ] **(D)** It remains strictly zero at all temperatures because 1D chains cannot absorb thermal energy.

In [16]:
ws.render_section('q22_13_boundary_condition_invariance')

**Q13 · medium** · 💡

Consider solving the 1D Ising chain with open boundary conditions ($s_1$ and $s_N$ have only one neighbor) versus periodic boundary conditions ($s_{N+1} = s_1$). How does the choice of boundary conditions affect the free energy per spin in the thermodynamic limit $N \to \infty$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Consider the fraction of total bonds that are modified when transitioning from periodic to open boundaries in a chain of length N.

</div>
</details>

- [ ] **(A)** The free energy per spin is strictly identical in the thermodynamic limit because the boundary contributes an energy of order $\mathcal{O}(1)$, which vanishes when divided by $N$ as $N \to \infty$.
- [ ] **(B)** Open boundary conditions induce a finite-temperature phase transition, whereas periodic boundaries prevent it.
- [ ] **(C)** Periodic boundaries double the critical temperature relative to open boundaries.
- [ ] **(D)** The free energy per spin diverges under open boundaries due to boundary spin unbinding.

In [17]:
ws.render_section('sec_onsager_solution_and_critical_phenomena')

### Kramers-Wannier Duality and Onsager's Critical Point
In 1941, Hendrik Kramers and Gregory Wannier discovered an exact duality relating the high-temperature and low-temperature expansions of the 2D square-lattice Ising model on dual lattices. High-temperature graphs of $\tanh(\beta J)$ map directly onto low-temperature domain wall graphs of $e^{-2\beta^* J}$. The self-dual transformation condition:
$$\sinh(2\beta J) \sinh(2\beta^* J) = 1$$
identifies the unique self-dual point $\beta = \beta^* = \beta_c$, which must be the critical point if only a single phase transition exists:
$$\sinh(2\beta_c J) = 1 \implies 2\beta_c J = \ln(1 + \sqrt{2}) \approx 0.8813736$$
Inverting this yields Lars Onsager's celebrated 1944 exact critical temperature:
$$\frac{k_B T_c}{J} = \frac{2}{\ln(1 + \sqrt{2})} \approx 2.269185$$

### Spontaneous Magnetization and Critical Exponents
Onsager announced and C. N. Yang (1952) rigorously derived the exact spontaneous magnetization below $T_c$:
$$M_0(T) = \left[ 1 - \frac{1}{[\sinh(2\beta J)]^4} \right]^{1/8} \quad (T < T_c)$$
Near $T_c$, expanding $\sinh(2\beta J)$ yields $M_0(T) \propto (T_c - T)^{\beta}$ with the exact order-parameter critical exponent $\beta = 1/8 = 0.125$. The complete set of exact 2D Ising critical exponents is:
- **Specific Heat:** $C_V \propto -\ln|1 - T/T_c| \implies \alpha = 0$ (logarithmic divergence)
- **Spontaneous Magnetization:** $M_0 \propto (T_c - T)^{\beta} \implies \beta = 1/8 = 0.125$
- **Isothermal Susceptibility:** $\chi_T \propto |T - T_c|^{-\gamma} \implies \gamma = 7/4 = 1.75$
- **Correlation Length:** $\xi \propto |T - T_c|^{-\nu} \implies \nu = 1$
- **Critical Correlation Decay:** $G(r) \propto r^{-(d - 2 + \eta)} = r^{-\eta} \implies \eta = 1/4 = 0.25$

### Hyperscaling Relations and Breakdown of Mean-Field Theory
These exponents satisfy the universal scaling relations derived from the renormalization group:
- **Rushbrooke Equality:** $\alpha + 2\beta + \gamma = 0 + 2(1/8) + 7/4 = 2$
- **Josephson Hyperscaling:** $d\nu = 2 - \alpha \implies 2(1) = 2 - 0 = 2$
- **Fisher Relation:** $\gamma = \nu(2 - \eta) \implies 7/4 = 1(2 - 1/4) = 7/4$
By contrast, Landau Mean-Field Theory predicts $\alpha_{\text{MF}} = 0$ (step discontinuity), $\beta_{\text{MF}} = 1/2$, $\gamma_{\text{MF}} = 1$, and $\nu_{\text{MF}} = 1/2$. Mean-field theory completely fails in $d = 2$ because it neglects local thermal fluctuations. By the Ginzburg criterion, mean-field critical exponents only become quantitatively exact above the **upper critical dimension** $d_u = 4$.


In [18]:
ws.render_section('q22_14_kramers_wannier_duality')

**Q14 · hard** · 💡

How did Kramers and Wannier pinpoint the exact critical coupling of the 2D square-lattice Ising model prior to Onsager's full solution of the partition function?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Consider how a high-temperature expansion in terms of closed loop bonds on a lattice maps onto a low-temperature expansion in terms of domain wall boundaries on the dual lattice.

</div>
</details>

- [ ] **(A)** By establishing an exact mathematical duality relating the high-temperature expansion on the square lattice to the low-temperature domain wall expansion on its dual lattice, and identifying the self-dual fixed point $\sinh(2\beta_c J) = 1$.
- [ ] **(B)** By performing high-precision Monte Carlo simulations on mechanical vacuum tube computers.
- [ ] **(C)** By diagonalizing infinite-dimensional Toeplitz determinants using Wiener–Hopf techniques.
- [ ] **(D)** By applying the Hartree–Fock self-consistent field approximation to the spin density.

In [19]:
ws.render_section('q22_15_onsager_tc_numeric')

**Q15 · medium** · 💡

Onsager's exact analytical solution for the 2D square-lattice Ising ferromagnet establishes that the critical point is determined by the self-dual Kramers-Wannier relation $\sinh(2\beta_c J) = 1$, where $\beta_c = 1/(k_B T_c)$. Calculate the dimensionless critical temperature $k_B T_c / J$ to 3 decimal places.

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Use the inverse hyperbolic sine relation or express sinh(x) in terms of exponentials to solve for the critical argument 2*beta_c*J, and subsequently invert the thermal coupling to obtain the reduced temperature.

</div>
</details>

_Numeric answer required._

In [20]:
ws.render_section('q22_16_spontaneous_magnetization_exponent')

**Q16 · medium** · 💡

Below $T_c$, Yang's exact formula demonstrates that the spontaneous magnetization vanishes as $M_0(T) \propto (T_c - T)^{\beta}$. How does the exact 2D critical exponent $\beta$ compare to the value predicted by classical Mean-Field Theory?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall Yang's eighth-root formula for spontaneous magnetization versus the square-root dependence derived from truncating the Landau free energy at fourth order.

</div>
</details>

- [ ] **(A)** $\beta = 1/8 = 0.125$ in the 2D Ising model, which is four times smaller than the mean-field prediction of $\beta_{\text{MF}} = 1/2 = 0.50$, causing magnetization to rise much more sharply below $T_c$.
- [ ] **(B)** $\beta = 1/2$ in both 2D and mean-field theory because critical exponents are strictly dimension-independent.
- [ ] **(C)** $\beta = 1/4$ in 2D and $\beta_{\text{MF}} = 1/8$ in mean-field theory.
- [ ] **(D)** $\beta = 1$ in 2D, indicating a linear decrease of magnetization to zero.

In [21]:
ws.render_section('q22_17_2d_critical_exponents_multiselect')

**Q17 · hard** · 💡

Which of the following statements correctly specify the exact critical exponents and scaling properties of the 2D Ising universality class? *(Select all that apply.)*

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check each exponent against Onsager's exact solution and verify whether the hyperscaling relation d*nu = 2 - alpha is satisfied.

</div>
</details>

- [ ] **(A)** The correlation length exponent is $\nu = 1$, meaning the spatial correlation length diverges as $\xi \sim |T - T_c|^{-1}$.
- [ ] **(B)** The magnetic susceptibility exponent is $\gamma = 7/4 = 1.75$, far stronger than the mean-field divergence of $\gamma_{\text{MF}} = 1$.
- [ ] **(C)** The specific heat exponent $\alpha = 0$ corresponds to a symmetric logarithmic divergence $C_V \sim -\ln|T - T_c|$ rather than a power law.
- [ ] **(D)** The critical correlation function at $T = T_c$ decays as $G(r) \sim r^{-1/4}$, giving anomalous dimension $\eta = 1/4$.
- [ ] **(E)** The upper critical dimension is $d_u = 2$, so mean-field exponents are exact in two dimensions.

In [22]:
ws.render_section('q22_18_rushbrooke_scaling_law')

**Q18 · medium** · 💡

Thermodynamic stability requires that the magnetic susceptibility, specific heat, and spontaneous magnetization satisfy the Rushbrooke scaling relation. What is the universal algebraic relationship between the critical exponents $\alpha$, $\beta$, and $\gamma$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Differentiate the singular part of the free energy scaling hypothesis f_sing ~ |T - Tc|^(2 - alpha) twice with respect to temperature and magnetic field.

</div>
</details>

- [ ] **(A)** $\alpha + 2\beta + \gamma = 2$
- [ ] **(B)** $\alpha + \beta + \gamma = 1$
- [ ] **(C)** $\alpha \beta \gamma = 2$
- [ ] **(D)** $2\alpha + \beta - \gamma = 0$

In [23]:
ws.render_section('q22_19_upper_critical_dimension')

**Q19 · medium** · 💡

According to the Ginzburg criterion, what is the upper critical dimension $d_u$ for the Ising universality class, and what physical change occurs when spatial dimension $d \ge d_u$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Consider the spatial dimension above which order-parameter fluctuations become negligible compared to mean-field thermal averages over a correlation volume xi^d.

</div>
</details>

- [ ] **(A)** $d_u = 4$; above four dimensions, thermal fluctuations on length scales $\xi$ become negligible over correlation volumes, rendering Landau mean-field critical exponents exact.
- [ ] **(B)** $d_u = 2$; above two dimensions, phase transitions cease to exist.
- [ ] **(C)** $d_u = 3$; systems in three dimensions always have mean-field exponents.
- [ ] **(D)** $d_u = \infty$; fluctuations dominate in all finite dimensions.

In [24]:
ws.render_section('sec_order_parameter_distribution_and_interfaces')

### The Order Parameter Distribution $P_L(M)$
In a finite lattice of linear dimension $L$ ($N = L^d$ spins), the total magnetization per spin is $M = \frac{1}{N}\sum_{i=1}^N s_i$. The probability density $P_L(M)$ provides a direct window into the finite-size Landau free energy landscape $\mathcal{F}_L(M) = -k_B T \ln P_L(M)$:
1. **Disordered Phase ($T > T_c$):** Thermal fluctuations are uncorrelated across distances greater than $\xi \ll L$. By the Central Limit Theorem, $P_L(M)$ is a single Gaussian centered at $M = 0$ with variance $\sigma_M^2 = k_B T \chi_T / N$.
2. **Critical Point ($T = T_c$):** Fluctuations become scale-invariant ($\xi \sim L$). The Gaussian quadratic coefficient vanishes, and $P_L(M)$ flattens into a non-Gaussian distribution governed by the quartic term:
$$P_L(M) \propto \exp\left( -u L^{d - 4\beta/\nu} M^4 \right)$$
3. **Ordered Phase ($T < T_c$):** Spontaneous symmetry breaking splits $P_L(M)$ into **two symmetric peaks** centered near $\pm M_0(T)$. In a finite box with $h = 0$, exact $\mathbb{Z}_2$ inversion symmetry ensures $\langle M \rangle = 0$, but the peaks are separated by a deep free energy barrier at $M = 0$.

### Interfacial Tension and Periodic Domain Walls
To sample configurations with $M \approx 0$ at $T < T_c$, the simulation box must form coexistence between $+M_0$ and $-M_0$ phases. Under periodic boundary conditions, topological closure requires an even number of interfaces: a minimum of **two flat domain walls** of cross-sectional area $L^{d-1}$ spanning the periodic boundary. The excess free energy penalty is:
$$\Delta \mathcal{F}_L = 2 \sigma(T) L^{d-1}$$
where $\sigma(T)$ is the interfacial free energy per unit area. The probability ratio between the minimum at $M = 0$ and the peaks at $\pm M_0$ scales as:
$$\frac{P_L(0)}{P_L(M_0)} \propto \exp\left( -\frac{\Delta \mathcal{F}_L}{k_B T} \right) = \exp\left( -\frac{2 \sigma(T) L^{d-1}}{k_B T} \right)$$
As $L \to \infty$, this barrier diverges, causing spontaneous ergodicity breaking in the thermodynamic limit.

### The Binder Fourth-Order Cumulant
Kurt Binder (1981) introduced a dimensionless cumulant that cleanly locates critical points independently of unknown non-universal constants:
$$U_4(T, L) = 1 - \frac{\langle M^4 \rangle_L}{3 \langle M^2 \rangle_L^2}$$
- At $T \gg T_c$, Gaussian statistics yield $\langle M^4 \rangle = 3 \langle M^2 \rangle^2 \implies U_4 \to 0$.
- At $T \ll T_c$, bimodal delta peaks at $\pm M_0$ yield $\langle M^4 \rangle = \langle M^2 \rangle^2 = M_0^4 \implies U_4 \to 2/3$.
- Right at $T = T_c$, by finite-size scaling, $U_4(T_c, L) = U^*$ is **invariant with respect to $L$** (up to corrections to scaling). Plotting $U_4(T, L)$ versus $T$ for various system sizes produces curves that intersect at a single invariant crossing point identifying $T_c$.


In [25]:
ws.render_section('concept_ising_magnetization_pdf')

### Interactive Concept: Finite-Size Magnetization PDF & Landau Free Energy Landscape

Explore how the order parameter distribution $P_L(M)$ and Landau free energy $\beta F(M) = -\ln P_L(M)$ evolve across temperatures, lattice sizes $L$, and external magnetic fields $h$.

**Key Insight:** Below $T_c$, $P_L(M)$ splits into two broken-symmetry peaks separated by a free energy barrier scaling as $2\sigma L^{d-1}$. Applying a small magnetic field $h > 0$ tilts the potential via the Zeeman term $-N h M$, exponentially suppressing the minority $-M_0$ peak by a factor $e^{-2\beta N h M_0}$.

In [26]:
ws.render_section('q22_20_periodic_boundary_two_domain_walls')

**Q20 · medium** · 💡

In a simulation cell with periodic boundary conditions, why does traversing the configuration space from a state dominated by positive magnetization to one dominated by negative magnetization require the creation of at least two domain walls rather than a single interface?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Consider the topology of a flat torus formed by periodic boundaries, and trace any continuous closed coordinate path that crosses an embedded slab of reversed spins.

</div>
</details>

- [ ] **(A)** Because periodic boundary conditions identify coordinate $x = 0$ with $x = L$; any slab of reversed spins embedded in the bulk must have both an entry interface ($+ \to -$) and an exit interface ($- \to +$) to satisfy cyclic continuity.
- [ ] **(B)** Because spins are only permitted to flip in pairs by the Pauli exclusion principle.
- [ ] **(C)** Because single domain walls violate energy conservation during Monte Carlo steps.
- [ ] **(D)** Because the transfer matrix has two distinct eigenvalues.

In [27]:
ws.render_section('q22_21_interfacial_free_energy_numeric')

**Q21 · medium** · 💡

In a Monte Carlo simulation of a 2D square-lattice Ising ferromagnet on an $L \times L$ grid with periodic boundary conditions ($L = 48$), the system is cooled to an ordered state where the dimensionless interfacial free energy per unit length is $\beta \sigma = 0.125$. Under periodic boundaries, the minimum free energy configuration at zero net magnetization ($M = 0$) consists of two flat system-spanning domain walls. Calculate the total dimensionless free energy barrier $\beta \Delta \mathcal{F}_L$ separating the two degenerate ferromagnetic ground states.

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Account for the cross-sectional length of each system-spanning interface in two dimensions and the total number of independent interfaces mandated by periodic boundary conditions.

</div>
</details>

_Numeric answer required._

In [28]:
ws.render_section('q22_22_zeeman_field_exponential_suppression')

**Q22 · medium** · 💡

When an external magnetic field $h > 0$ is applied to an Ising ferromagnet below $T_c$ in the interactive concept visualizer, how does the ratio of probabilities between the two broken-symmetry states, $P_L(+M_0) / P_L(-M_0)$, scale with the total number of lattice sites $N$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Evaluate the Zeeman energy difference Delta E_h between a state with total magnetization +N*M_0 and a state with total magnetization -N*M_0 in the canonical Boltzmann distribution.

</div>
</details>

- [ ] **(A)** It scales exponentially as $\exp(2\beta N h M_0)$, completely suppressing the minority $-M_0$ state in macroscopic systems even for infinitesimal positive fields.
- [ ] **(B)** It scales linearly with $N$ as $(1 + 2\beta h M_0)$.
- [ ] **(C)** It remains equal to $1$ because discrete $\mathbb{Z}_2$ symmetry cannot be broken by a magnetic field.
- [ ] **(D)** It decreases as $1/N$ due to finite-size rounding.

In [29]:
ws.render_section('q22_23_binder_cumulant_crossing_method')

**Q23 · hard** · 💡

Why is the Binder fourth-order cumulant $U_4(T, L) = 1 - \frac{\langle M^4 \rangle_L}{3 \langle M^2 \rangle_L^2}$ one of the most powerful computational tools for locating continuous phase transitions in finite-size Monte Carlo simulations?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Consider how the dimensionless ratio of moments behaves under finite-size scaling when the correlation length xi is replaced by system size L right at T = Tc.

</div>
</details>

- [ ] **(A)** Because $U_4$ is a dimensionless ratio whose leading-order scaling function is scale-invariant at $T = T_c$, causing curves of $U_4(T, L)$ for different box sizes $L$ to intersect at a common crossing point that pinpoints $T_c$.
- [ ] **(B)** Because $U_4$ is strictly zero at all temperatures except exactly at $T = T_c$ where it diverges to infinity.
- [ ] **(C)** Because $U_4$ eliminates the need to perform Monte Carlo sampling by computing the partition function analytically.
- [ ] **(D)** Because $U_4$ only requires measuring single-spin flip probabilities.

In [30]:
ws.render_section('q22_24_susceptibility_finite_size_scaling')

**Q24 · hard** · 💡

In an infinite system, the magnetic susceptibility diverges as $\chi_T \propto |T - T_c|^{-\gamma}$. In a finite 2D simulation cell of linear dimension $L$, how does the maximum peak value of the susceptibility $\chi_{\max}(L)$ scale with $L$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Apply finite-size scaling: replace the diverging correlation length xi ~ |T - Tc|^(-nu) with the system size cutoff L in the susceptibility divergence relation.

</div>
</details>

- [ ] **(A)** $\chi_{\max}(L) \sim L^{\gamma/\nu} = L^{1.75}$
- [ ] **(B)** $\chi_{\max}(L) \sim L^0 = \text{constant}$
- [ ] **(C)** $\chi_{\max}(L) \sim L^2$
- [ ] **(D)** $\chi_{\max}(L) \sim e^{-L}$

In [31]:
ws.render_section('q22_25_fluctuation_dissipation_susceptibility')

**Q25 · medium** · 💡

In a canonical Monte Carlo simulation of an Ising model at zero external field ($h = 0$), how is the static isothermal susceptibility $\chi_T = \lim_{h \to 0} \frac{\partial \langle M \rangle}{\partial h}$ computed from sampled equilibrium configurations?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Differentiate the canonical ensemble average of magnetization with respect to external field h using the definition of the partition function.

</div>
</details>

- [ ] **(A)** Via the fluctuation-dissipation theorem relating response to variance: $\chi_T = \frac{N}{k_B T}(\langle M^2 \rangle - \langle M \rangle^2)$.
- [ ] **(B)** By computing the time derivative of total energy $dE/dt$.
- [ ] **(C)** By evaluating the square root of the mean magnetization $\sqrt{\langle M \rangle}$.
- [ ] **(D)** From the inverse of the transfer matrix trace $1 / \operatorname{Tr}(\mathbf{T}^N)$.

In [32]:
ws.render_section('sec_spin_kinetics_coarsening_and_ripening')

### Non-Equilibrium Quenches and Phase Separation Kinetics
When an Ising system is rapidly quenched from a high-temperature disordered state ($T \gg T_c$) to below $T_c$, the uniform state becomes unstable. Microscopic domains of up and down spins nucleate, coalesce, and grow through **coarsening** (domain growth). The kinetic mechanism and growth law depend profoundly on whether the order parameter is conserved:

### Glauber Dynamics (Model A: Non-Conserved Order Parameter)
- **Elementary Move:** A single spin $s_i$ is randomly selected and flipped: $s_i \to -s_i$.
- **Conservation:** The total magnetization $M = \frac{1}{N}\sum s_i$ is **not conserved** ($\dot{M} \ne 0$).
- **Physical Context:** Models magnetic domains in ferromagnets where atomic magnetic moments rotate under local molecular fields.
- **Lifshitz-Allen-Cahn Growth Law:** Domain wall velocity is driven strictly by local interface curvature $\kappa$:
$$v_n = -\Gamma \sigma \kappa$$
Because the characteristic curvature scales inversely with domain radius $\kappa \sim 1/L(t)$ and velocity is $dL/dt$, we have:
$$\frac{dL}{dt} \sim \frac{1}{L} \implies L(t)^2 - L(0)^2 \propto t \implies L(t) \propto t^{1/2}$$

### Kawasaki Dynamics (Model B: Conserved Order Parameter)
- **Elementary Move:** Two nearest-neighbor spins with opposite signs ($s_i = +1, s_j = -1$) are **exchanged**: $s_i \leftrightarrow s_j$.
- **Conservation:** The total magnetization $M = \frac{1}{N}\sum s_i$ (i.e. atomic composition) is **strictly conserved** at its initial value.
- **Physical Context:** Models binary alloy phase separation (precipitation, spinodal decomposition) and liquid-gas demixing, where solute atoms must diffuse through the lattice.
- **Lifshitz-Slyozov-Wagner (LSW) Growth Law:** Because mass is conserved, domains cannot grow simply by local boundary movement. Solute must diffuse through the matrix from shrinking high-curvature droplets to larger low-curvature droplets (**Ostwald ripening**). The chemical potential gradient drives a diffusive flux scaling as:
$$\frac{dL}{dt} \sim \frac{D \sigma}{L^2} \implies L(t)^3 - L(0)^3 \propto t \implies L(t) \propto t^{1/3}$$
Mass conservation suppresses domain coarsening from $t^{1/2}$ down to $t^{1/3}$.

### Dynamical Scaling and Porod's Law
During late-stage coarsening, the domain morphology exhibits statistical self-similarity governed by the **dynamical scaling hypothesis**. The equal-time structure factor $S(k, t) = \langle |\hat{s}(\mathbf{k}, t)|^2 \rangle$ collapses onto a universal scaling function:
$$S(k, t) = L(t)^d F(k L(t))$$
At large wavevectors ($k L(t) \gg 1$), scattering from sharp, thin interfaces produces the universal power-law tail known as **Porod's Law**:
$$S(k, t) \propto \Sigma(t) k^{-(d+1)}$$
where $\Sigma(t) \propto 1/L(t)$ is the total interfacial area per unit volume. In $d = 2$, $S(k) \propto k^{-3}$; in $d = 3$, $S(k) \propto k^{-4}$.


In [33]:
ws.render_section('concept_glauber_vs_kawasaki_coarsening')

### Interactive Concept: Domain Coarsening (Glauber t^(1/2) vs. Kawasaki t^(1/3))

Compare non-conserved Glauber dynamics with conserved Kawasaki dynamics following a deep quench to $T = 0.6 T_c$. Switch between kinetic modes, adjust quench time $t$, and observe domain morphology and characteristic length scale $R(t)$.

**Key Insight:** Glauber dynamics rapidly collapses minority islands via curvature-driven boundary motion ($L(t) \sim t^{1/2}$), allowing the majority phase to homogenize the box. Kawasaki dynamics strictly conserves composition, forcing the system to maintain both phases in an interconnected bicontinuous labyrinth (spinodal decomposition) that coarsens through slow diffusive mass transfer ($L(t) \sim t^{1/3}$).

In [34]:
ws.render_section('q22_26_glauber_vs_kawasaki_conservation')

**Q26 · easy** · 💡

What fundamental conservation constraint distinguishes Kawasaki spin kinetics from Glauber spin kinetics, and what class of physical phenomena does each model represent?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compare the elementary trial move of each algorithm: flipping a single spin versus swapping two nearest-neighbor spins.

</div>
</details>

- [ ] **(A)** Kawasaki kinetics strictly conserves total magnetization $M = \frac{1}{N}\sum_i s_i$ (modeling diffusive mass transport and phase separation in binary alloys), whereas Glauber kinetics allows $M$ to vary freely (modeling domain wall motion in magnetic materials).
- [ ] **(B)** Kawasaki kinetics conserves total energy, whereas Glauber kinetics operates in the microcanonical ensemble.
- [ ] **(C)** Glauber kinetics strictly conserves momentum, whereas Kawasaki kinetics includes hydrodynamic turbulence.
- [ ] **(D)** Kawasaki dynamics is deterministic Newton's motion, whereas Glauber dynamics is stochastic Brownian motion.

In [35]:
ws.render_section('q22_27_allen_cahn_curvature_scaling')

**Q27 · medium** · 💡

In non-conserved (Glauber / Model A) domain coarsening, what physical driving force governs the migration velocity of domain walls, and what temporal growth law $L(t)$ does it produce?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Relate interface velocity to local curvature kappa ~ 1/L, and integrate the resulting first-order differential equation for domain size L(t).

</div>
</details>

- [ ] **(A)** Interface velocity is driven by local mean curvature $v_n = -\Gamma \sigma \kappa \sim -1/L$, yielding the parabolic Lifshitz-Allen-Cahn growth law $L(t) \sim t^{1/2}$.
- [ ] **(B)** Interface velocity is driven by bulk chemical potential gradients, producing linear growth $L(t) \sim t$.
- [ ] **(C)** Interface velocity is limited by long-range volume diffusion, producing $L(t) \sim t^{1/3}$.
- [ ] **(D)** Interface velocity is logarithmic $L(t) \sim \ln t$ due to random pinning impurities.

In [36]:
ws.render_section('q22_28_kawasaki_domain_growth_numeric')

**Q28 · medium** · 💡

A binary alloy undergoing phase separation under conserved Kawasaki spin-exchange kinetics (Model B) is monitored during isothermal annealing. At an early observation time $t_1 = 50\text{ MCS}$, the average precipitate domain size is measured to be $L_1 = 6.0\text{ nm}$. Assuming domain growth is governed by the universal Lifshitz-Slyozov-Wagner scaling regime, calculate the expected domain size $L_2$ (in nm) at time $t_2 = 400\text{ MCS}$.

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall the universal temporal growth power-law exponent characteristic of conserved order parameter dynamics driven by long-range diffusive transport between domains, and apply that scaling ratio between the two observation times.

</div>
</details>

_Numeric answer required (in nm)._

In [37]:
ws.render_section('q22_29_glauber_domain_growth_numeric')

**Q29 · medium** · 💡

In a second experiment, a magnetic thin film is quenched below its Curie temperature and coarsens via non-conserved Glauber single-spin-flip dynamics (Model A). If the average magnetic domain diameter is measured to be $L_1 = 4.0\text{ nm}$ at $t_1 = 25\text{ MCS}$, calculate the expected domain diameter $L_2$ (in nm) at $t_2 = 225\text{ MCS}$ under the Lifshitz-Allen-Cahn curvature-driven growth regime.

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Consider how the growth exponent differs when domain boundary migration is governed strictly by local interface curvature rather than mass conservation, and scale the domain size by the corresponding time ratio.

</div>
</details>

_Numeric answer required (in nm)._

In [38]:
ws.render_section('q22_30_ostwald_ripening_gibbs_thomson')

**Q30 · medium** · 💡

Why does Ostwald ripening occur during late-stage conserved phase separation, and what thermodynamic relation establishes the chemical potential driving force?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall how boundary curvature affects the local equilibrium solute solubility or vapor pressure at a curved interface.

</div>
</details>

- [ ] **(A)** By the Gibbs-Thomson relation $\mu(R) = \mu_{\text{flat}} + \frac{2\gamma\Omega}{R}$, smaller droplets have higher interfacial curvature and elevated surface chemical potential, creating a concentration gradient that drives diffusive flux toward larger droplets.
- [ ] **(B)** Gravitational settling forces smaller particles to sink and aggregate onto larger particles.
- [ ] **(C)** Smaller domains have lower surface-to-volume ratio, making them thermodynamically more stable.
- [ ] **(D)** Kawasaki dynamics introduces an artificial mechanical pressure that squeezes small domains.

In [39]:
ws.render_section('q22_31_spinodal_vs_nucleation')

**Q31 · hard** · 💡

In the phase diagram of a binary mixture or Ising lattice gas below $T_c$, what thermodynamic criterion separates the metastable regime (nucleation and growth) from the spontaneously unstable regime (spinodal decomposition)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Examine the sign of the second derivative of the local free energy density f(M) with respect to composition or magnetization.

</div>
</details>

- [ ] **(A)** The spinodal line where $\frac{\partial^2 f}{\partial M^2} = 0$; inside the spinodal ($\frac{\partial^2 f}{\partial M^2} < 0$), the homogeneous phase is barrierlessly unstable to infinitesimal long-wavelength concentration fluctuations.
- [ ] **(B)** The binodal line where chemical potentials equalize, regardless of curvature.
- [ ] **(C)** The zero-entropy contour $S = 0$.
- [ ] **(D)** The line where the latent heat of transition vanishes.

In [40]:
ws.render_section('q22_32_porods_law_scattering')

**Q32 · hard** · 💡

In X-ray or neutron scattering experiments monitoring late-stage phase separation in a $d$-dimensional material, Porod's law describes the asymptotic power-law decay of the structure factor $S(k, t)$ at high wavevectors ($k L(t) \gg 1$). What is the exact mathematical power law in spatial dimension $d$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Consider Fourier transforming the step-discontinuity across sharp domain interfaces of dimension d-1 embedded in d-dimensional space.

</div>
</details>

- [ ] **(A)** $S(k, t) \sim k^{-(d + 1)}$, yielding $k^{-3}$ in 2D and $k^{-4}$ in 3D.
- [ ] **(B)** $S(k, t) \sim k^{-d}$, yielding $k^{-2}$ in 2D and $k^{-3}$ in 3D.
- [ ] **(C)** $S(k, t) \sim \exp(-k^2 L(t)^2)$
- [ ] **(D)** $S(k, t) \sim k^{d - 1}$

In [41]:
ws.render_section('sec_critical_slowing_down_and_cluster_mc')

### Critical Slowing Down in Local Spin-Flip Algorithms
Near a continuous phase transition ($T \approx T_c$), spatial correlations extend over macroscopically large domains of size $\xi \propto |T - T_c|^{-\nu}$. In standard single-spin-flip algorithms (Metropolis or Glauber), propagating information across a correlated domain requires a diffusive random walk of domain walls. Consequently, the **autocorrelation time** $\tau$ diverges as:
$$\tau \propto \xi^z \propto |T - T_c|^{-z\nu}$$
where $z$ is the **dynamic critical exponent**. In a finite simulation box at $T = T_c$ where $\xi \sim L$, $\tau \propto L^z$. For local single-spin-flip Metropolis in 2D, $z \approx 2.17$. Generating an independent configuration requires $\mathcal{O}(\tau)$ sweeps, so the total computational effort per independent sample scales as:
$$\text{CPU Effort} \sim L^d \times L^z = L^{d + z} \approx L^{4.17} \quad (d = 2)$$
Doubling the lattice size from $L = 64$ to $L = 128$ increases sampling cost by a factor of $2^{4.17} \approx 18\times$. Simulating large systems near $T_c$ with local flips is severely bottlenecked by this **critical slowing down**.

### The Fortuin-Kasteleyn Mapping and Swendsen-Wang Cluster Algorithm
In 1987, Robert Swendsen and Jian-Sheng Wang solved this computational bottleneck by mapping the Ising spin model onto the **Fortuin-Kasteleyn (FK) random cluster model**. Instead of flipping single spins, correlated clusters of spins are identified and updated simultaneously.

**Swendsen-Wang Algorithm Procedure:**
1. **Bond Activation:** Inspect all nearest-neighbor pairs $\langle i, j \rangle$. If the spins are parallel ($s_i = s_j$), activate a virtual bond between them with probability:
$$p = 1 - e^{-2\beta J}$$
If the spins are antiparallel ($s_i \ne s_j$), no bond can form ($p = 0$).
2. **Cluster Identification:** Find all connected components of sites linked by active bonds using a linear-time graph labeling algorithm (Disjoint-Set / Union-Find with path compression, or Hoshen-Kopelman).
3. **Cluster Spin Flip:** For each connected cluster independently, pick a new spin orientation $s_{\text{cluster}} \in \{-1, +1\}$ with equal probability $1/2$ (for $h = 0$). Assign this spin orientation to all sites in that cluster.

### Wolff Single-Cluster Algorithm
In 1989, Ulli Wolff introduced a single-cluster variation that avoids decomposing the entire lattice:
1. Select a random seed site $i$ on the lattice.
2. Recursively grow a cluster around $i$ by inspecting all neighboring sites $j$ of current cluster sites. If $s_j = s_i$ and the bond has not been tested, add site $j$ to the cluster with probability $p = 1 - e^{-2\beta J}$.
3. Invert all spins in the cluster: $s_k \to -s_k$.

Because large clusters have more sites, a randomly selected seed site is chosen with probability proportional to cluster size, automatically focusing computational effort where it is most needed. Cluster algorithms dramatically slash the dynamic critical exponent in 2D from $z \approx 2.17$ down to $z_{\text{SW}} \approx 0.35$ (and $z_{\text{Wolff}} \approx 0.25$), virtually eradicating critical slowing down!


In [42]:
ws.render_section('concept_swendsen_wang_clusters')

### Interactive Concept: Swendsen-Wang Fortuin-Kasteleyn Clusters & Dynamic Exponent

Visualize the Fortuin-Kasteleyn bond creation probability $p = 1 - e^{-2\beta J}$, percolation threshold $p_c$, and the dramatic reduction in critical autocorrelation time $\tau(L)$ achieved by cluster updates.

**Key Insight:** At $T \gg T_c$, bond probability is near zero ($p \to 0$), recovering independent random spin updates. At $T \ll T_c$, $p \to 1$, grouping nearly the whole lattice into a single cluster that flips collectively. Right at $T = T_c$, bonded clusters percolate fractally with the exact Ising correlation length $\xi$, reducing critical autocorrelation time from $\tau \sim L^{2.17}$ down to $\tau \sim L^{0.25}$!

In [43]:
ws.render_section('q22_33_critical_slowing_down_origin')

**Q33 · medium** · 💡

Why does the autocorrelation time $\tau$ of local single-spin-flip Monte Carlo algorithms diverge near continuous phase transitions (critical slowing down)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Consider how information about a spin flip propagates across a spatial distance equal to the diverging correlation length xi when updates are strictly local.

</div>
</details>

- [ ] **(A)** Because spatial correlations diverge ($\xi \sim |T - T_c|^{-\nu}$); rearranging a correlated domain of size $\xi$ using local single-spin flips requires a diffusive random walk of domain walls, causing relaxation time to scale as $\tau \sim \xi^z$.
- [ ] **(B)** Because the pseudorandom number generator runs out of entropy at $T_c$.
- [ ] **(C)** Because the acceptance probability of single spin flips drops to zero.
- [ ] **(D)** Because the system runs out of accessible microstates.

In [44]:
ws.render_section('q22_34_fk_critical_bond_probability_numeric')

**Q34 · medium** · 💡

In the Fortuin-Kasteleyn representation underlying the Swendsen-Wang and Wolff cluster algorithms, adjacent parallel spins are dynamically connected by bonds with probability $p = 1 - e^{-2\beta J}$. At the exact 2D square-lattice critical coupling where $\sinh(2\beta_c J) = 1$, calculate the numerical value of the critical bond activation probability $p_c$ to 4 decimal places.

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Relate exp(-2*beta_c*J) to the algebraic value of exp(2*beta_c*J) obtained from the critical hyperbolic condition sinh(2*beta_c*J) = 1, and evaluate the resulting bond activation probability.

</div>
</details>

_Numeric answer required._

In [45]:
ws.render_section('q22_35_swendsen_wang_spin_assignment')

**Q35 · medium** · 💡

In the Swendsen-Wang algorithm, once all connected clusters have been identified via active Fortuin-Kasteleyn bonds at zero external field ($h = 0$), how is the new spin configuration generated?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Consider how global discrete Z_2 symmetry applies independently to each disconnected subgraph in the random cluster representation.

</div>
</details>

- [ ] **(A)** For each connected cluster independently, a new collective spin orientation ($+1$ or $-1$) is chosen with equal probability $1/2$ and assigned to all sites in that cluster.
- [ ] **(B)** All clusters are inverted deterministically to their opposite orientation.
- [ ] **(C)** Only the single largest cluster is flipped, while all smaller clusters retain their old spins.
- [ ] **(D)** Spins inside clusters are randomized site-by-site without regard to cluster connectivity.

In [46]:
ws.render_section('q22_36_wolff_seed_selection_bias')

**Q36 · hard** · 💡

Why is the Wolff single-cluster algorithm often more efficient in practice than the full Swendsen-Wang multi-cluster algorithm?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Consider how picking a single random lattice site as a seed affects the probability that a cluster of size C is selected for growth and flipping.

</div>
</details>

- [ ] **(A)** Selecting a random seed site automatically selects clusters with probability proportional to their size $C / N$, focusing computational effort on flipping large, dominant clusters while avoiding the overhead of decomposing tiny 1-site clusters in disordered regions.
- [ ] **(B)** Wolff algorithm does not require calculating the bond probability $p$.
- [ ] **(C)** Wolff algorithm operates entirely without random numbers.
- [ ] **(D)** Wolff algorithm completely avoids cluster recursion.

In [47]:
ws.render_section('q22_37_cluster_speedup_numeric')

**Q37 · hard** · 💡

In a 2D Ising simulation performed directly at the critical point $T_c$, single-spin Metropolis dynamics exhibits an autocorrelation time scaling as $\tau_{\text{local}} \sim L^{z_{\text{local}}}$ with $z_{\text{local}} \approx 2.17$, while the Swendsen-Wang cluster algorithm exhibits $\tau_{\text{SW}} \sim L^{z_{\text{SW}}}$ with $z_{\text{SW}} \approx 0.35$. On a lattice of linear size $L = 64$, calculate the theoretical algorithmic speedup ratio $\tau_{\text{local}} / \tau_{\text{SW}}$ in autocorrelation time.

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Express the speedup ratio as a power of the lattice dimension using the difference between the two dynamic critical exponents.

</div>
</details>

_Numeric answer required._

In [48]:
ws.render_section('q22_38_cluster_algorithm_limitations')

**Q38 · hard** · 💡

Under which physical conditions do the Swendsen-Wang and Wolff cluster algorithms LOSE their immense efficiency advantage over local Metropolis sampling? *(Select all that apply.)*

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Consider how the presence of external magnetic fields or competing frustrated interactions affects cluster growth and the cluster acceptance/flip criteria.

</div>
</details>

- [ ] **(A)** In the presence of a strong external magnetic field $h \ne 0$, where clusters grow to encompass the entire box and collective flips are heavily penalized by the Zeeman energy.
- [ ] **(B)** In geometrically frustrated systems (e.g., antiferromagnets on triangular lattices or spin glasses) with competing positive and negative exchange bonds.
- [ ] **(C)** In standard zero-field ferromagnetic models near $T_c$.
- [ ] **(D)** When measuring the Binder cumulant at the crossing point.

In [49]:
ws.render_section('q22_39_union_find_complexity')

**Q39 · medium** · 💡

What data structure and algorithm is standardly used to identify all connected components in Swendsen-Wang in near-linear time $\mathcal{O}(N \alpha(N))$, where $\alpha$ is the inverse Ackermann function?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall the classic disjoint-set forest data structure equipped with union-by-rank and path compression heuristics.

</div>
</details>

- [ ] **(A)** Disjoint-Set (Union-Find) with path compression and union-by-rank, or the Hoshen-Kopelman algorithm.
- [ ] **(B)** Dijkstra's single-source shortest path algorithm using a Fibonacci heap.
- [ ] **(C)** Floyd-Warshall all-pairs shortest path matrix decomposition.
- [ ] **(D)** Fast Fourier Transform spectral clustering.

In [50]:
ws.render_section('q22_40_fortuin_kasteleyn_percolation_mapping')

**Q40 · hard** · 💡

What is the mathematical connection between the Fortuin-Kasteleyn random cluster representation of the Ising model and standard random bond percolation?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Notice that when the number of states q in the Potts/random-cluster model is taken to q = 1, the cluster weight simplifies to uncorrelated bond probabilities.

</div>
</details>

- [ ] **(A)** The Ising model corresponds to the $q = 2$ random cluster model where each connected component carries a statistical weight factor of 2, whereas standard uncorrelated percolation corresponds to the mathematical limit $q \to 1$.
- [ ] **(B)** Standard percolation only occurs in one dimension, whereas the Ising model only exists in two dimensions.
- [ ] **(C)** The Ising model and percolation are completely unrelated and share no mathematical similarities.
- [ ] **(D)** Fortuin-Kasteleyn mapping replaces spins with continuous Gaussian random walks.

In [51]:
ws.render_section('q22_41_porod_law_scattering_tail')

**Q41 · hard** · 💡

In a phase-separating mixture or coarsening Ising domain structure characterized by sharp, smooth spatial interfaces, how does the high-wavevector tail of the spherically averaged structure factor $S(k, t)$ decay with wavenumber $k$ in $d$ spatial dimensions (Porod's law)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Consider the Fourier transform of a spatial correlation function across an abrupt step-discontinuity interface of finite total surface area.

</div>
</details>

- [ ] **(A)** $S(k, t) \propto k^{-(d+1)}$ as $k \to \infty$
- [ ] **(B)** $S(k, t) \propto k^{-(d-1)}$ as $k \to \infty$
- [ ] **(C)** $S(k, t) \propto e^{-k^2 \xi^2}$ as $k \to \infty$
- [ ] **(D)** $S(k, t) \propto k^{-d/2}$ as $k \to \infty$

In [52]:
ws.render_section('q22_42_metropolis_autocorrelation_scaling_numeric')

**Q42 · medium** · 💡

In a 2D Ising simulation right at the critical temperature $T_c$ using local single-spin Metropolis updates ($z = 2.17$), the integrated autocorrelation time for a lattice of linear dimension $L = 64$ is measured to be $\tau = 10,000\text{ sweeps}$. Based on dynamical finite-size scaling, predict the autocorrelation time in sweeps when the lattice size is doubled to $L = 128$. *(Round to the nearest 1000 sweeps.)*

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Scale the baseline autocorrelation time by the system dimension expansion ratio raised to the dynamical critical exponent.

</div>
</details>

_Numeric answer required._

In [ ]:
ws.show_record()

In [ ]:
ws.finalize()